In [0]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 76.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os


print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:29<00:00, 155MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-94fb4922-aaa9-4597-8086-1b nogroup 8.4G Jan 23 06:27 2019-Nov.csv
-rwxrwxrwx 1 spark-94fb4922-aaa9-4597-8086-1b nogroup 5.3G Jan 23 06:30 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 23 06:19 delta
-rwxrwxrwx 1 spark-94fb4922-aaa9-4597-8086-1b nogroup 4.3G Jan 23 06:27 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 23 06:19 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-94fb4922-aaa9-4597-8086-1b nogroup 8.4G Jan 23 06:27 2019-Nov.csv
-rwxrwxrwx 1 spark-94fb4922-aaa9-4597-8086-1b nogroup 5.3G Jan 23 06:30 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 23 06:19 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 23 06:19 outputs


In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
csv_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events_oct2019"

db = "workspace.ecommerce"
table_managed = f"{db}.events_oct2019_managed"
table_external = f"{db}.events_oct2019_external"

events = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(csv_path))

events = (events
          .withColumn("price", F.col("price").cast("double")))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_gold")
base_vol = "/Volumes/workspace/ecommerce/ecommerce_data"

raw_csv = f"{base_vol}/2019-Oct.csv"

bronze_path = f"{base_vol}/delta/bronze/events"
silver_path = f"{base_vol}/delta/silver/events"
gold_path   = f"{base_vol}/delta/gold/product_perf"


In [0]:
bronze = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(raw_csv)
          .withColumn("ingestion_ts", F.current_timestamp())
          .withColumn("source_file", F.lit(raw_csv)))

(bronze.write
 .format("delta")
 .mode("overwrite")
 .save(bronze_path))

print("Bronze written to:", bronze_path)

Bronze written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events


In [0]:
(bronze.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_bronze.events"))

In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

silver = (bronze_df
          .withColumn("event_ts", F.to_timestamp("event_time"))
          .withColumn("event_date", F.to_date("event_ts"))
          .withColumn("price", F.col("price").cast("double"))
          .filter(F.col("event_ts").isNotNull())
          .filter(F.col("event_type").isNotNull())
          .filter(F.col("user_session").isNotNull())
          .filter((F.col("price").isNull()) | ((F.col("price") > 0) & (F.col("price") < 10000)))
          .dropDuplicates(["user_session", "event_time", "event_type", "product_id"])
          .withColumn(
              "price_tier",
              F.when(F.col("price").isNull(), F.lit("unknown"))
               .when(F.col("price") < 10, F.lit("budget"))
               .when(F.col("price") < 50, F.lit("mid"))
               .otherwise(F.lit("premium"))
          )
         )

(silver.write
 .format("delta")
 .mode("overwrite")
 .save(silver_path))

print("Silver written to:", silver_path)

Silver written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events


In [0]:
(silver.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_silver.events"))

In [0]:
silver_df = spark.read.format("delta").load(silver_path)

product_perf = (
    silver_df.groupBy("product_id")
    .agg(
        F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("unique_viewers"),
        F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("unique_purchasers"),
        F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))), 2).alias("revenue")
    )
    .withColumn(
        "conversion_rate_pct",
        F.when(F.col("unique_viewers") == 0, F.lit(0.0))
         .otherwise(F.round((F.col("unique_purchasers") / F.col("unique_viewers")) * 100, 4))
    )
)

(product_perf.write
 .format("delta")
 .mode("overwrite")
 .save(gold_path))

print("Gold written to:", gold_path)

display(product_perf.orderBy(F.col("revenue").desc()).limit(20))

Gold written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf


product_id,unique_viewers,unique_purchasers,revenue,conversion_rate_pct
1005115,170989,8352,1.240483595E7,4.8845
1005105,114813,4794,1.023924868E7,4.1755
1004249,96989,5538,6729380.83,5.7099
1005135,62646,2163,5567806.64,3.4527
1004767,175572,14410,5430222.72,8.2075
1002544,89025,6781,4854785.55,7.617
1004856,197840,19228,3798168.71,9.719
1002524,51704,4132,3538299.12,7.9916
1003317,56575,2179,3051294.26,3.8515
1004870,84318,7331,3027098.05,8.6945


In [0]:
(product_perf.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("workspace.ecommerce_gold.product_perf"))

In [0]:
dbutils.widgets.text("source_csv", "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv", "Source CSV")
dbutils.widgets.text("bronze_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events", "Bronze path")
dbutils.widgets.text("bronze_table", "workspace.ecommerce_bronze.events", "Bronze table")

source_csv  = dbutils.widgets.get("source_csv")
bronze_path = dbutils.widgets.get("bronze_path")
bronze_table= dbutils.widgets.get("bronze_table")

In [0]:
raw = (spark.read.format("csv")
       .option("header","true")
       .option("inferSchema","true")
       .load(source_csv)
       .withColumn("ingestion_ts", F.current_timestamp())
       .withColumn("source_file", F.lit(source_csv)))

(raw.write.format("delta").mode("overwrite").save(bronze_path))
(raw.write.format("delta").mode("overwrite").saveAsTable(bronze_table))

print("Bronze done:", bronze_path, bronze_table)

Bronze done: /Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events workspace.ecommerce_bronze.events


In [0]:
dbutils.widgets.text("bronze_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/bronze/events", "Bronze path")
dbutils.widgets.text("silver_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events", "Silver path")
dbutils.widgets.text("silver_table", "workspace.ecommerce_silver.events", "Silver table")

bronze_path = dbutils.widgets.get("bronze_path")
silver_path = dbutils.widgets.get("silver_path")
silver_table= dbutils.widgets.get("silver_table")

In [0]:
bronze = spark.read.format("delta").load(bronze_path)

silver = (bronze
          .withColumn("event_ts", F.to_timestamp("event_time"))
          .withColumn("event_date", F.to_date("event_ts"))
          .withColumn("price", F.col("price").cast("double"))
          .filter(F.col("event_ts").isNotNull())
          .filter(F.col("user_session").isNotNull())
          .filter((F.col("price").isNull()) | ((F.col("price") > 0) & (F.col("price") < 10000)))
          .dropDuplicates(["user_session", "event_time", "event_type", "product_id"])
         )

(silver.write.format("delta").mode("overwrite").save(silver_path))
(silver.write.format("delta").mode("overwrite").saveAsTable(silver_table))

print("Silver done:", silver_path, silver_table)

Silver done: /Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events workspace.ecommerce_silver.events


In [0]:
dbutils.widgets.text("silver_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/silver/events", "Silver path")
dbutils.widgets.text("gold_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf", "Gold path")
dbutils.widgets.text("gold_table", "workspace.ecommerce_gold.product_perf", "Gold table")

silver_path = dbutils.widgets.get("silver_path")
gold_path   = dbutils.widgets.get("gold_path")
gold_table  = dbutils.widgets.get("gold_table")

In [0]:
silver = spark.read.format("delta").load(silver_path)

gold = (silver.groupBy("product_id")
        .agg(
            F.countDistinct(F.when(F.col("event_type") == "view", F.col("user_id"))).alias("unique_viewers"),
            F.countDistinct(F.when(F.col("event_type") == "purchase", F.col("user_id"))).alias("unique_purchasers"),
            F.round(F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))), 2).alias("revenue")
        )
        .withColumn(
            "conversion_rate_pct",
            F.when(F.col("unique_viewers") == 0, F.lit(0.0))
             .otherwise(F.round((F.col("unique_purchasers") / F.col("unique_viewers")) * 100, 4))
        )
       )

(gold.write.format("delta").mode("overwrite").save(gold_path))
(gold.write.format("delta").mode("overwrite").saveAsTable(gold_table))

print("Gold done:", gold_path, gold_table)

Gold done: /Volumes/workspace/ecommerce/ecommerce_data/delta/gold/product_perf workspace.ecommerce_gold.product_perf


In [0]:
from pyspark.sql.window import Window

silver_table = "workspace.ecommerce_silver.events"
events = spark.table(silver_table)

In [0]:
events.printSchema()
print("Total rows:", events.count())

display(
    events.groupBy("event_type")
          .count()
          .orderBy(F.col("count").desc())
)

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- price_tier: string (nullable = true)

Total rows: 42349871


event_type,count
view,40708806
cart,898292
purchase,742773


In [0]:
events.select("price").describe().show()
display(
    events.selectExpr(
        "percentile_approx(price, 0.5) as p50",
        "percentile_approx(price, 0.9) as p90",
        "percentile_approx(price, 0.99) as p99"
    )
)

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          42349871|
|   mean| 290.7840021246416|
| stddev|358.39679231746146|
|    min|              0.77|
|    max|           2574.07|
+-------+------------------+



p50,p90,p99
163.92,743.62,1741.34


In [0]:
events2 = (
    events
    .withColumn("event_ts", F.to_timestamp("event_time"))
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("dow", F.dayofweek("event_date"))         
    .withColumn("is_weekend", F.col("dow").isin([1, 7]))   
)

display(events2.select("event_time", "event_ts", "event_date", "dow", "is_weekend").limit(5))

event_time,event_ts,event_date,dow,is_weekend
2019-10-13T06:25:59.000Z,2019-10-13T06:25:59.000Z,2019-10-13,1,true
2019-10-13T06:26:40.000Z,2019-10-13T06:26:40.000Z,2019-10-13,1,true
2019-10-13T06:28:24.000Z,2019-10-13T06:28:24.000Z,2019-10-13,1,true
2019-10-13T06:28:28.000Z,2019-10-13T06:28:28.000Z,2019-10-13,1,true
2019-10-13T06:28:36.000Z,2019-10-13T06:28:36.000Z,2019-10-13,1,true


In [0]:
session_conv = (
    events2.groupBy("user_session", "is_weekend")
           .agg(
               F.max(F.when(F.col("event_type") == "purchase", F.lit(1)).otherwise(F.lit(0))).alias("converted")
           )
)

display(session_conv.groupBy("is_weekend", "converted").count().orderBy("is_weekend","converted"))
display(session_conv.groupBy("is_weekend").agg(F.avg("converted").alias("conversion_rate")).orderBy("is_weekend"))

is_weekend,converted,count
false,0,6364784
false,1,462905
true,0,2257028
true,1,166699


is_weekend,conversion_rate
false,0.06779819643220422
true,0.06877796055413832


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import ChiSquareTest

chi_input = (
    session_conv
    .select(
        F.col("converted").cast("double").alias("label"),
        F.col("is_weekend").cast("double").alias("is_weekend_num")
    )
)

va = VectorAssembler(inputCols=["is_weekend_num"], outputCol="features")
chi_ready = va.transform(chi_input).select("label", "features")

chi = ChiSquareTest.test(chi_ready, "features", "label").head()
print("pValues:", chi.pValues, "degreesOfFreedom:", chi.degreesOfFreedom, "statistics:", chi.statistics)

pValues: [1.958755044828564e-07] degreesOfFreedom: [1] statistics: [27.073392418908238]


pValues: [1.958755044828564e-07] degreesOfFreedom: [1] statistics: [27.073392418908238]


In [0]:
events_corr = (
    events2
    .filter(F.col("price").isNotNull())
    .withColumn("is_purchase", (F.col("event_type") == "purchase").cast("double"))
    .select(F.col("price").cast("double").alias("price"), "is_purchase")
)

print("corr(price, is_purchase) =", events_corr.corr("price", "is_purchase"))

corr(price, is_purchase) = 0.006999963921127964


In [0]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

num_df = (
    events2
    .filter(F.col("price").isNotNull())
    .withColumn("hour", F.hour("event_ts").cast("double"))
    .withColumn("dow_num", F.col("dow").cast("double"))
    .withColumn("price_log", F.log(F.col("price") + F.lit(1.0)))
    .withColumn("is_purchase", (F.col("event_type") == "purchase").cast("double"))
    .select("hour", "dow_num", "price_log", "is_purchase")
)

va2 = VectorAssembler(inputCols=["hour", "dow_num", "price_log", "is_purchase"], outputCol="features")
feat_df = va2.transform(num_df).select("features")

corr_mat = Correlation.corr(feat_df, "features", "pearson").head()[0]
print(corr_mat)

DenseMatrix([[ 1.00000000e+00, -1.14076579e-04,  8.53327834e-03,
              -2.39070642e-02],
             [-1.14076579e-04,  1.00000000e+00,  6.18967718e-05,
               3.42255733e-04],
             [ 8.53327834e-03,  6.18967718e-05,  1.00000000e+00,
               1.39670651e-02],
             [-2.39070642e-02,  3.42255733e-04,  1.39670651e-02,
               1.00000000e+00]])


In [0]:
w_user = Window.partitionBy("user_id").orderBy("event_ts")
w_session = Window.partitionBy("user_session").orderBy("event_ts")

features = (
    events2
    .filter(F.col("event_ts").isNotNull())
    .withColumn("hour", F.hour("event_ts"))
    .withColumn("day_of_week", F.dayofweek("event_date"))  
    .withColumn("price_clean", F.coalesce(F.col("price").cast("double"), F.lit(0.0)))
    .withColumn("price_log", F.log(F.col("price_clean") + F.lit(1.0)))

    .withColumn("first_event_ts_user", F.first("event_ts").over(w_user))
    .withColumn("time_since_first_event_user_sec",
                F.unix_timestamp("event_ts") - F.unix_timestamp("first_event_ts_user"))

    .withColumn("event_index_in_session", F.row_number().over(w_session))
    .withColumn("prior_event_ts_session", F.lag("event_ts", 1).over(w_session))
    .withColumn("time_since_prev_event_session_sec",
                F.unix_timestamp("event_ts") - F.unix_timestamp("prior_event_ts_session"))
)

display(features.select(
    "user_id","user_session","event_ts","event_type",
    "hour","day_of_week","price_clean","price_log",
    "event_index_in_session","time_since_prev_event_session_sec",
    "time_since_first_event_user_sec"
).limit(10))

user_id,user_session,event_ts,event_type,hour,day_of_week,price_clean,price_log,event_index_in_session,time_since_prev_event_session_sec,time_since_first_event_user_sec
559841837,00001417-945d-4ab1-a69c-3e6a13b186f2,2019-10-14T19:03:55.000Z,view,19,2,252.21,5.534219183960909,1,null,99625
514294709,00006c6c-799c-4f14-baf2-3ebd863da937,2019-10-22T16:51:37.000Z,view,16,3,90.09,4.511848028756718,1,null,871379
513409103,000089a1-ee47-49e1-9450-91f7c2e3155f,2019-10-26T10:51:18.000Z,view,10,7,138.74,4.939783553124305,1,null,1636392
515841602,00009359-f7d4-49cc-b8eb-839702a78d3e,2019-10-17T07:37:27.000Z,view,7,5,28.06,3.369362658142137,1,null,1272542
515841602,00009359-f7d4-49cc-b8eb-839702a78d3e,2019-10-17T07:38:44.000Z,view,7,5,28.06,3.369362658142137,2,77,1272619
515841602,00009359-f7d4-49cc-b8eb-839702a78d3e,2019-10-17T07:39:53.000Z,view,7,5,28.06,3.369362658142137,3,69,1272688
515841602,00009359-f7d4-49cc-b8eb-839702a78d3e,2019-10-17T07:40:08.000Z,view,7,5,25.1,3.261935314328648,4,15,1272703
515841602,00009359-f7d4-49cc-b8eb-839702a78d3e,2019-10-17T07:41:01.000Z,view,7,5,59.3,4.09933210373314,5,53,1272756
515841602,00009359-f7d4-49cc-b8eb-839702a78d3e,2019-10-17T07:42:28.000Z,view,7,5,59.3,4.09933210373314,6,87,1272843
519529265,0000f5f2-31c5-411c-9c48-de82e0cde146,2019-10-18T02:32:15.000Z,view,2,6,79.78,4.391729410135267,1,null,461703


In [0]:
session_features = (
    features.groupBy("user_session", "is_weekend")
            .agg(
                F.max((F.col("event_type") == "purchase").cast("int")).alias("label_converted"),
                F.count("*").alias("events_in_session"),
                F.sum((F.col("event_type") == "view").cast("int")).alias("views_in_session"),
                F.sum((F.col("event_type") == "cart").cast("int")).alias("carts_in_session"),
                F.sum((F.col("event_type") == "purchase").cast("int")).alias("purchases_in_session"),
                F.max("price_clean").alias("max_price_seen"),
                F.avg("price_clean").alias("avg_price_seen"),
                F.max("event_index_in_session").alias("session_length_events"),
                F.max("time_since_first_event_user_sec").alias("user_age_sec_at_session_end")
            )
)

display(session_features.limit(20))

user_session,is_weekend,label_converted,events_in_session,views_in_session,carts_in_session,purchases_in_session,max_price_seen,avg_price_seen,session_length_events,user_age_sec_at_session_end
00001417-945d-4ab1-a69c-3e6a13b186f2,false,0,1,1,0,0,252.21,252.21,1,99625
00006c6c-799c-4f14-baf2-3ebd863da937,false,0,1,1,0,0,90.09,90.09,1,871379
000089a1-ee47-49e1-9450-91f7c2e3155f,true,0,1,1,0,0,138.74,138.74,1,1636392
00009359-f7d4-49cc-b8eb-839702a78d3e,false,0,6,6,0,0,59.3,37.98,6,1272843
0000a854-51d0-4f87-8859-92d600b31986,false,0,1,1,0,0,25.15,25.15,1,0
0000f5f2-31c5-411c-9c48-de82e0cde146,false,0,6,6,0,0,79.78,52.98666666666668,6,462344
00011dd8-4265-4edb-8c02-28a6b3c9b1df,false,0,13,13,0,0,161.91,102.16153846153846,13,78422
000135e4-4f91-4cf6-a6c2-7f4c85cf09c3,false,0,6,6,0,0,2557.59,1179.7816666666668,6,712
00015b16-9528-4066-9bbf-c1da047d060b,false,0,2,2,0,0,1428.31,944.405,2,865669
00018506-6d94-4b52-b460-57178da05621,false,0,2,2,0,0,746.22,746.22,2,16
